# Q-learning propagates future value

Q-learning updates an action using its immediate reward plus the estimated value of the next state. First use the standard FrozenLake environment from Gymnasium, then build a five-state environment and Q-table directly.

## 1. Q-learning in a Gymnasium environment
Use the standard deterministic FrozenLake environment while keeping the Q-table visible.

In [ ]:
import sys, subprocess
try:
    import gymnasium as gym
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gymnasium'])
    import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2)
env = gym.make('FrozenLake-v1', is_slippery=False)
env.action_space.seed(2)
Q = np.zeros((env.observation_space.n, env.action_space.n))
alpha, gamma, epsilon = .2, .95, .3
history = []

for episode in range(5000):
    state, _ = env.reset(seed=episode)
    done = False
    while not done:
        action = env.action_space.sample() if rng.random() < epsilon else Q[state].argmax()
        next_state, reward, terminated, truncated, _ = env.step(action)
        target = reward + gamma * Q[next_state].max() * (not terminated)
        Q[state, action] += alpha * (target - Q[state, action])
        state = next_state
        done = terminated or truncated
    history.append(Q[0].max())

print('Start-state action values:', Q[0].round(3))
print('Greedy action by state:', Q.argmax(axis=1))
plt.plot(history); plt.xlabel('episode'); plt.ylabel('estimated value at start'); plt.title('Future reward propagates back to state 0'); plt.show();

## 2. Build the same routine from scratch
Use only Python lists and `random` so every part of the Q-learning update remains visible.

In [ ]:
import random

random.seed(2)
Q_scratch = [[0.0, 0.0] for _ in range(5)]
for episode in range(1000):
    state = 0
    while state < 4:
        explore = random.random() < .2
        action = random.randrange(2) if explore else max(range(2), key=lambda a: Q_scratch[state][a])
        next_state = min(4, state + (2 if action else 1))
        reward = 1.0 if next_state == 4 else -.02
        target = reward + .95 * max(Q_scratch[next_state])
        Q_scratch[state][action] += .2 * (target - Q_scratch[state][action])
        state = next_state

for state, values in enumerate(Q_scratch):
    print(state, [round(value, 3) for value in values])

## What to notice
State 0 receives no terminal reward directly. Its value rises because its actions lead to states from which reward is reachable. That continuation value is the defining ingredient missing from a one-step decision.